# CYTools-agent

Demos the CYTools-agent code. Runs specialized CYTools tools (`fetch_polytopes`, `get_polytope_info`) with a local Ollama model.

Run with the **Python (cytools-agent)** kernel (`./setup.sh` creates it).

## Setup — Ollama client

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "qwen2.5-coder:7b-instruct"
assert MODEL in [m.id for m in client.models.list().data], f"{MODEL} not pulled"
print("OK:", MODEL)

## Tools — automatic schemas

Generate each OpenAI tool schema from the function's signature and docstring.

In [ ]:
from cytools_agent.tools import polytope as pt
from cytools_agent.schema import function_to_schema
import json

TOOL_FNS   = [pt.fetch_polytopes, pt.get_polytope_info]
tools      = [function_to_schema(fn) for fn in TOOL_FNS]
tool_impls = {fn.__name__: fn for fn in TOOL_FNS}

print(json.dumps(tools[0]["function"]["parameters"], indent=2))

## Loop

The fallback parser (recovers tool JSON when Qwen drops the `<tool_call>` markers) and `run_agent` are unchanged generic machinery from the coding-agent tutorial.

In [ ]:
import json

def extract_tool_call(content, known_tools):
    if not content:
        return None

    try:
        obj = json.loads(content.strip().replace("\x00", ""))
    except json.JSONDecodeError:
        return None
    if not isinstance(obj, dict) or obj.get("name") not in known_tools:
        return None
    args = obj.get("arguments")
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            return None
    return {"name": obj["name"], "arguments": args} if isinstance(args, dict) else None

def extract_tool_call(content, known_tools):
    if not content: 
        return None
    
    # strip bad characters
    text = content.strip().replace("\x00", "")
    
    # read first real JSON object, if there are multiple
    if text.startswith("```"):                        # strip a ```json ... ``` fence
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    try:  
        obj, _ = json.JSONDecoder().raw_decode(text)  # first JSON object, ignore the rest
    except json.JSONDecodeError:
        return None 

    # check that the object corresponds to a real tool
    if not isinstance(obj, dict) or obj.get("name") not in known_tools:
          return None
    args = obj.get("arguments")

    # decode string args
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            return None

    # return
    if isinstance(args, dict):
        # dict - expected
        return {"name": obj["name"], "arguments": args}
    else:
        # non-dict... fails
        return None

def run_agent(user_message, tools, tool_impls, max_steps=10, verbosity=0):
    messages = [
        {"role": "system", "content":
        "You are a CYTools research assistant. Call ONE tool at a time and WAIT for "
        "its result before deciding the next call. Only pass ks_ind values that "
        "fetch_polytopes actually returned — never placeholders."},
        {"role": "user", "content": user_message},
    ]
    for step in range(max_steps):
        msg = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools).choices[0].message
        messages.append(msg)

        # parse the message type
        if msg.tool_calls:
            if verbosity >= 1: print("run_agent: Tool call")

            calls = [(c.id, c.function.name, json.loads(c.function.arguments))
                     for c in msg.tool_calls]
        elif (fb := extract_tool_call(msg.content, set(tool_impls))):
            if verbosity >= 1: print("run_agent: (Recovered) tool call")

            calls = [(f"fallback_{step}", fb["name"], fb["arguments"])]
        else:
            if verbosity >= 1: print("run_agent: Text message")

            return msg.content

        # run the tools
        for call_id, name, args in calls:
            try:
                result = tool_impls[name](**args)
            except Exception as e:
                result = f"ERROR: {e}"
            messages.append({"role": "tool", "tool_call_id": call_id,
                             "content": str(result)})
    return "(max_steps exceeded)"

## Demo

In [ ]:
answer = run_agent(
    "Fetch 3 polytopes at h11=5, then report the h21 and number of points of each",
    tools=tools,
    tool_impls=tool_impls,
    verbosity=1
)
print(answer)